In [ ]:
##   start from inference

In [ ]:
import pathlib
import joblib
import pandas as pd
import numpy as np
from datetime import datetime
import yaml


import rest_api.schemas
import features.util_feature
import features.feature_engineer

E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\src\domain\usstates.json
True

E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor\src\domain\source_types.json



In [2]:
request_data = {
  "capacity": 250.0,
  "capacity_factor": 0.65,
  "activity": 120000.0,
  "source_type": "coal",
  "state": "Georgia",
  "area": 153910,
  "pop2020": 10711908
}

pRequest_df = pd.DataFrame( [request_data] )
# pRequest_df = pd.DataFrame( [request_data.dict()] )
pRequest_df

,capacity,capacity_factor,activity,source_type,state,area,pop2020
0,250.0,0.65,120000.0,coal,Georgia,153910,10711908


In [3]:
# BASE_DIR = pathlib.Path(__file__).resolve().parents[2]
BASE_DIR = pathlib.Path( r'E:\Learning_course\MLOps\coursera_packt\greenHouse_Emission_predictor' )
MODEL_DIR = BASE_DIR / 'models' / 'trained'

MODEL_PATH = MODEL_DIR / 'greenhouse_emission_predict_model.pkl'
PREPROCESSOR_PATH = MODEL_DIR / 'preprocessor.pkl'
MODEL_CONFIG_PATH = BASE_DIR / 'configs' / 'model_config.yaml'

In [4]:
model = joblib.load(MODEL_PATH)
preprocessor = joblib.load(PREPROCESSOR_PATH)

In [ ]:

with open(MODEL_CONFIG_PATH, 'r', encoding='utf-8') as f:
    model_cfg = yaml.safe_load(f)

In [9]:
### feature engg


# Apply EXACT SAME feature engineering as training
featured_df = features.util_feature.create_features(pRequest_df)
print( featured_df.shape )

# Apply saved preprocessor (OHE with fixed category space)
# xT = preprocessor.transform(featured_df)

# Build the same engineered feature table as in feature_engineer.py
engineered_x_df, engineered_cols = features.util_feature.transform_to_engineered_df(
    preprocessor=preprocessor,
    xx= featured_df,
    remaining_features_ls= features.feature_engineer.REMAINING_Features_ls 
    )

engineered_x_df

2026-01-13 22:48:28,145 - create features - INFO - Creating new features
2026-01-13 22:48:28,154 - create features - INFO - Created 1.transforms + ratios  2.power-system structure  3. Interaction features.


(1, 23)


,state_Alabama,state_Alaska,state_Arizona,state_Arkansas,state_California,state_Colorado,state_Connecticut,state_Delaware,state_Florida,state_Georgia,...,activity_per_capita,activity_per_area,capacity_per_capita,capacity_density,potential_output,utilization_ratio,activity_capacityFactor,activity_per_capacity,activity_capacity,capacity_factor_capacity
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.011202,0.779676,0.000023,0.001624,162.5,738.461538,78000.0,480.0,663054.352696,3.591544


In [17]:
yhat = int( model.predict(engineered_x_df.values)[0] )

confidence_interval = [ round( yhat * 0.9, 0 ),  round( yhat * 1.1, 0) ]
confidence_interval


[232621.0, 284315.0]